In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_sales = spark.table("workspace.google_drive.silver_restaurant_master_sales")

***Dimensions Creation***

In [0]:
dim_product = (silver_sales
    .select("category", "item_name", "price")
    .distinct()
    .withColumn("product_id", F.md5(F.concat(F.col("category"), F.col("item_name"))))
)

dim_product.write.format("delta").mode("overwrite").saveAsTable("workspace.google_drive.gold_dim_product")

In [0]:
dim_branch = (silver_sales
    .select("branch")
    .distinct()
    .withColumn("branch_id", F.md5(F.col("branch")))
)

dim_branch.write.format("delta").mode("overwrite").saveAsTable("workspace.google_drive.gold_dim_branch")

In [0]:
dim_customer = (silver_sales
    .select("customer_id")
    .distinct()
    .filter("customer_id IS NOT NULL")
    .withColumn("customer_segment", 
        F.when(F.col("customer_id") % 2 == 0, "Loyal").otherwise("Regular"))
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable("workspace.google_drive.gold_dim_customer")

In [0]:
from pyspark.sql.functions import explode, sequence, to_date, min, max

date_range = spark.table("workspace.google_drive.silver_restaurant_master_sales") \
    .select(min("order_date").alias("min_date"), max("order_date").alias("max_date")) \
    .collect()

start_date = date_range[0]['min_date']
end_date = date_range[0]['max_date']

dim_date_dynamic = spark.sql(f"""
    SELECT
        datum AS date_key,
        year(datum) AS year,
        month(datum) AS month,
        date_format(datum, 'MMMM') AS month_name,
        day(datum) AS day,
        dayofweek(datum) AS day_of_week,
        date_format(datum, 'EEEE') AS day_name,
        quarter(datum) AS quarter,
        CASE WHEN dayofweek(datum) IN (1, 7) THEN True ELSE False END AS is_weekend
    FROM (
        SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS datum
    )
""")

dim_date_dynamic.write.format("delta").mode("overwrite").saveAsTable("workspace.google_drive.gold_dim_date")

print(f"Dim_Date created dynamically from {start_date} to {end_date}")

Dim_Date created dynamically from 2020-01-01 to 2025-12-30


In [0]:
dim_order_attr = (silver_sales
    .select("payment_method", "order_type")
    .distinct()
    .withColumn("attr_id", F.md5(F.concat(F.col("payment_method"), F.col("order_type"))))
)

dim_order_attr.write.format("delta").mode("overwrite").saveAsTable("workspace.google_drive.gold_dim_order_attr")

***Fact Table***

In [0]:
from pyspark.sql import functions as F

silver_sales = spark.table("workspace.google_drive.silver_restaurant_master_sales")


gold_fact_sales = (silver_sales
    .withColumn("product_id", F.md5(F.concat(F.col("category"), F.col("item_name"))))
    .withColumn("branch_id", F.md5(F.col("branch")))
    .withColumn("attr_id", F.md5(F.concat(F.col("payment_method"), F.col("order_type"))))
    
    
    .select(
        "source_order_id",      # unique PK
        "order_date",           # FK Dim_Date
        "hour",                 
        "product_id",           # FK Dim_Product
        "branch_id",            # FK  Dim_Branch
        "attr_id",              # FK  Junk Dimension (Attributes)
        "customer_id",          # FK  Dim_Customer
        "quantity",
        "price",
        "discount",
        "total_amount",
        "rating"
    )
)

(gold_fact_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.google_drive.gold_fact_sales"))

#optimizationnnnn
spark.sql("OPTIMIZE workspace.google_drive.gold_fact_sales ZORDER BY (order_date, branch_id)")

print("New Fact Table is ready with all Foreign Keys!")

New Fact Table is ready with all Foreign Keys!


***Aggregated Gold Tables***

In [0]:
daily_branch_metrics = (gold_fact_sales
    .groupBy("order_date", "branch_id")
    .agg(
        F.sum("total_amount").alias("daily_revenue"),
        F.countDistinct("source_order_id").alias("daily_orders_count"),
        F.avg("rating").alias("avg_daily_rating")
    )
)
daily_branch_metrics.write.format("delta").mode("overwrite").saveAsTable("workspace.google_drive.gold_daily_branch_metrics")

In [0]:
peak_hours_analysis = (gold_fact_sales
    .groupBy("hour", "branch_id")
    .agg(F.count("source_order_id").alias("hourly_traffic"))
)
peak_hours_analysis.write.format("delta").mode("overwrite").saveAsTable("workspace.google_drive.gold_peak_hours")